In [1]:
from google.colab import files
uploaded = files.upload()

Saving Time Series Data.xlsx to Time Series Data.xlsx


In [2]:
import pandas as pd

# Replace with your actual file name after upload
file_name = list(uploaded.keys())[0]

df_original = pd.read_excel(file_name)

# Keep a copy of original data for evaluation
df_true = df_original.copy()

print("Data Shape:", df_original.shape)
df_original.head()

Data Shape: (5000, 8)


,US 10Y,China 10Y,Germany 10Y,UK 10Y,Japan 10Y,India 10Y,Gold,Silver
0,3.738,4.380,4.123,4.461,1.480,8.637,818.10,13.45
1,3.705,4.450,4.140,4.504,1.470,8.609,805.50,13.07
2,3.628,4.134,4.140,4.499,1.460,8.476,801.25,12.92
3,3.699,4.156,4.058,4.458,1.460,8.484,795.55,12.78
4,3.695,4.161,4.034,4.382,1.435,8.494,803.80,12.26


In [4]:
!pip install pykalman
import numpy as np
import pandas as pd
from pykalman import KalmanFilter
from sklearn.metrics import mean_squared_error, mean_absolute_error

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.1/252.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 12.0 MB/s eta 0:00:00


In [5]:
np.random.seed(42)

df_missing = df_original.copy()

missing_masks = {}

for col in df_missing.columns:
    data = df_missing[col].values.copy()

    n = len(data)
    missing_idx = np.random.choice(n, size=int(0.2 * n), replace=False)

    mask = np.zeros(n, dtype=bool)
    mask[missing_idx] = True

    data[mask] = np.nan

    df_missing[col] = data
    missing_masks[col] = mask

print("Missing data introduced successfully.")

Missing data introduced successfully.


In [6]:
df_imputed = df_missing.copy()

for col in df_missing.columns:
    series = df_missing[col].values

    # Mask missing values
    masked_series = np.ma.masked_invalid(series)

    # Define Kalman Filter (Local Level Model)
    kf = KalmanFilter(
        transition_matrices=[1],
        observation_matrices=[1],
        initial_state_mean=np.nanmean(series),
        initial_state_covariance=1,
        observation_covariance=1,
        transition_covariance=0.01
    )

    # Estimate parameters using EM
    kf = kf.em(masked_series, n_iter=10)

    # Apply smoothing
    state_means, _ = kf.smooth(masked_series)

    # Replace missing values with smoothed estimates
    imputed_series = series.copy()
    imputed_series[np.isnan(series)] = state_means[np.isnan(series), 0]

    df_imputed[col] = imputed_series

print("Kalman Imputation Completed.")

Kalman Imputation Completed.


In [8]:
results = []

for col in df_original.columns:
    true_values = df_true[col].values
    imputed_values = df_imputed[col].values
    mask = missing_masks[col]

    # Evaluate only where data was removed AND the original true value is not NaN
    y_true_masked = true_values[mask]
    y_pred_masked = imputed_values[mask]

    # Filter out NaNs from y_true_masked and corresponding y_pred_masked
    valid_indices = ~np.isnan(y_true_masked)

    y_true_valid = y_true_masked[valid_indices]
    y_pred_valid = y_pred_masked[valid_indices]

    # Only calculate metrics if there are valid points for comparison
    if len(y_true_valid) > 0:
        rmse = np.sqrt(mean_squared_error(y_true_valid, y_pred_valid))
        mae = mean_absolute_error(y_true_valid, y_pred_valid)
    else:
        # If all original values at masked locations were NaN, set metrics to NaN
        rmse = np.nan
        mae = np.nan

    results.append({
        "Column": col,
        "RMSE": rmse,
        "MAE": mae
    })

results_df = pd.DataFrame(results)

# Overall metrics
true_flat = df_true.values.flatten()
imputed_flat = df_imputed.values.flatten()

# Filter out NaNs from the true_flat array and corresponding imputed_flat values
valid_overall_indices = ~np.isnan(true_flat)

overall_rmse = np.sqrt(mean_squared_error(
    true_flat[valid_overall_indices],
    imputed_flat[valid_overall_indices]
))

overall_mae = mean_absolute_error(
    true_flat[valid_overall_indices],
    imputed_flat[valid_overall_indices]
)

print("Per Column Metrics:")
print(results_df)

print("\nOverall Metrics:")
print("RMSE:", overall_rmse)
print("MAE:", overall_mae)

Per Column Metrics:
        Column       RMSE        MAE
0       US 10Y   0.047799   0.035053
1    China 10Y   0.037979   0.022113
2  Germany 10Y   0.033768   0.025289
3       UK 10Y   0.044124   0.031083
4    Japan 10Y   0.018692   0.011086
5    India 10Y   0.045819   0.026420
6         Gold  18.773594  11.616997
7       Silver   0.901593   0.353545

Overall Metrics:
RMSE: 2.9603782645282015
MAE: 0.30074238771455786
